# Kurinda — Data Exploration

This notebook explores the five data sources that feed the Kurinda multi-source
fusion model:

1. **DHS 2019-20 Children's Recode** — childhood stunting outcome variable
2. **GADM v4.1 administrative boundaries** — geospatial backbone (422 sectors)
3. **CHIRPS rainfall** — monthly precipitation per sector (already extracted)
4. **MODIS NDVI** — monthly vegetation index per sector (already extracted)
5. **WFP food prices** — monthly retail food prices per district (already aggregated)

**Goal**: validate each dataset, sanity-check values against published statistics,
and understand the schema

## 0. Environment sanity check

In [14]:
import sys
print(f"Python:     {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")
print()

import pyreadstat
import pandas as pd
import geopandas as gpd

print(f"pyreadstat: {pyreadstat.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"geopandas:  {gpd.__version__}")

Python:     3.11.9
Executable: c:\Users\Hello\kurinda\backend\.venv\Scripts\python.exe

pyreadstat: 1.3.5
pandas:     3.0.3
geopandas:  1.1.3


## 1. Load the DHS Children's Recode (RWKR81DT)

One row per child under 5. Read `.DTA` directly from the zip (no extraction on disk — DHS terms of use)

In [15]:
import zipfile, tempfile, os
import pyreadstat
import pandas as pd
import numpy as np
from pathlib import Path

DHS_ZIP = Path("../../data/raw/dhs/2019-20/RWKR81DT.zip")
print(f"Exists: {DHS_ZIP.exists()} | Size: {DHS_ZIP.stat().st_size / 1e6:.2f} MB")

# List zip contents to confirm .DTA filename
with zipfile.ZipFile(DHS_ZIP) as zf:
    for n in zf.namelist():
        print(f"  {n}")

Exists: True | Size: 2.85 MB
  RWKR81FL.DCT
  RWKR81FL.DO
  RWKR81FL.DTA
  RWKR81FL.FRQ
  RWKR81FL.FRW
  RWKR81FL.MAP


In [16]:
DTA_NAME = "RWKR81FL.DTA"  # adjust if previous cell showed different name

with zipfile.ZipFile(DHS_ZIP) as zf:
    with zf.open(DTA_NAME) as src:
        with tempfile.NamedTemporaryFile(suffix=".DTA", delete=False) as tmp:
            tmp.write(src.read())
            tmp_path = tmp.name
try:
    df, meta = pyreadstat.read_dta(tmp_path)
finally:
    os.unlink(tmp_path)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Shape: 8,092 rows × 1,216 columns
Memory: 263.1 MB


## 2. Identify the 16 variables we need

Out of ~5,400 DHS columns, we keep: outcome (`HW70`), eligibility filters (`B5`, `HW1`), geography (`V001`, `V024`, `V025`), survey design (`V005`), and household predictors (`V190`, `V133`, `B4`).

In [17]:
KEY_VARS = {
    "hw70": "Height-for-age z-score (× 100)",
    "hw71": "Weight-for-age z-score (× 100)",
    "hw72": "Weight-for-height z-score (× 100)",
    "b5":   "Child is alive",
    "b8":   "Current age years",
    "hw1":  "Age in months at measurement",
    "v001": "Cluster (PSU) number",
    "v024": "Region",
    "v025": "Urban/rural",
    "v005": "Sample weight (÷ 1,000,000)",
    "v190": "Wealth quintile",
    "v133": "Mother's education years",
    "v012": "Mother's age",
    "b4":   "Child sex",
}

print(f"{'Var':<8s} {'OK':<4s} {'DHS Label':<60s}")
print("-" * 75)
for v, _ in KEY_VARS.items():
    ok = "✅" if v in df.columns else "❌"
    label = meta.column_names_to_labels.get(v, "(missing)")[:60]
    print(f"{v:<8s} {ok:<4s} {label}")

Var      OK   DHS Label                                                   
---------------------------------------------------------------------------
hw70     ✅    height/age standard deviation (new who)
hw71     ✅    weight/age standard deviation (new who)
hw72     ✅    weight/height standard deviation (new who)
b5       ✅    child is alive
b8       ✅    current age of child
hw1      ✅    child's age in months
v001     ✅    cluster number
v024     ✅    region
v025     ✅    type of place of residence
v005     ✅    women's individual sample weight (6 decimals)
v190     ✅    wealth index combined
v133     ✅    education in single years
v012     ✅    respondent's current age
b4       ✅    sex of child


## 3. Inspect raw distributions before deriving stunting

DHS encodes missing values as 9994-9999. We need to spot these before computing anything.

In [18]:
print("hw70 (HAZ × 100):")
print(df["hw70"].describe().round(1))
print(f"\nTop 5 values (look for 9996/9998 = missing codes):")
print(df["hw70"].value_counts().head(5))
print(f"\nb5 (alive): {dict(df['b5'].value_counts(dropna=False))}")
print(f"hw1 (age months) range: {df['hw1'].min()} to {df['hw1'].max()}")

hw70 (HAZ × 100):
count     3815
unique     591
top       -156
freq        24
Name: hw70, dtype: int64

Top 5 values (look for 9996/9998 = missing codes):
hw70
-156    24
-144    23
-174    23
-194    23
-126    23
Name: count, dtype: int64

b5 (alive): {1: np.int64(7796), 0: np.int64(296)}
hw1 (age months) range: 0 to 59


## 4. Derive stunting and validate against published DHS 2019-20 (33%)

WHO standard: stunted = HAZ < -2 SD. Apply DHS sample weights to estimate the national rate. Should land near 33%.

In [19]:
# Filter to children with valid HAZ
valid = (df["b5"] == 1) & (df["hw1"] <= 59) & (df["hw70"] < 9990)
children = df.loc[valid].copy()
children["HAZ"] = children["hw70"] / 100
children["stunted"] = (children["HAZ"] < -2).astype(int)
children["severely_stunted"] = (children["HAZ"] < -3).astype(int)
children["w"] = children["v005"] / 1_000_000

print(f"Children retained: {len(children):,} (of {len(df):,})")
print()

unweighted = children["stunted"].mean() * 100
weighted = (children["stunted"] * children["w"]).sum() / children["w"].sum() * 100
severe = (children["severely_stunted"] * children["w"]).sum() / children["w"].sum() * 100

print(f"Unweighted stunting rate: {unweighted:.1f}%")
print(f"Weighted stunting rate:   {weighted:.1f}%  ← compare to published 33.0%")
print(f"Severely stunted:         {severe:.1f}%")

Children retained: 3,809 (of 8,092)

Unweighted stunting rate: 33.4%
Weighted stunting rate:   33.0%  ← compare to published 33.0%
Severely stunted:         9.0%


## 5. Aggregate to cluster (PSU) level

DHS clusters are villages/neighborhoods where ~20-30 households were surveyed. Stunting rate per cluster is what we'll spatially join to GADM sectors in notebook 02.

In [20]:
def wmean(v, w):
    valid = v.notna() & w.notna()
    if valid.sum() == 0: return np.nan
    return (v[valid] * w[valid]).sum() / w[valid].sum()

cluster_agg = (
    children.groupby("v001")
    .apply(lambda g: pd.Series({
        "n_children":        len(g),
        "stunting_rate":     wmean(g["stunted"], g["w"]) * 100,
        "severe_rate":       wmean(g["severely_stunted"], g["w"]) * 100,
        "mean_haz":          wmean(g["HAZ"], g["w"]),
        "region":            g["v024"].iloc[0],
        "urban_rural":       g["v025"].iloc[0],
        "mean_wealth_q":     wmean(g["v190"], g["w"]),
        "mean_maternal_edu": wmean(g["v133"], g["w"]),
    }), include_groups=False)
    .reset_index()
    .rename(columns={"v001": "cluster_id"})
)

print(f"Clusters: {len(cluster_agg)}")
print(f"\nStunting rate distribution across clusters:")
print(cluster_agg["stunting_rate"].describe().round(1))
print(f"\nFirst 5 clusters:")
print(cluster_agg.head().to_string(index=False))

Clusters: 500

Stunting rate distribution across clusters:
count    500.0
mean      31.8
std       21.6
min        0.0
25%       15.4
50%       33.3
75%       50.0
max      100.0
Name: stunting_rate, dtype: float64

First 5 clusters:
 cluster_id  n_children  stunting_rate  severe_rate  mean_haz  region  urban_rural  mean_wealth_q  mean_maternal_edu
          1         6.0      33.333333     0.000000 -1.431667     1.0          2.0       4.666667           9.833333
          2         5.0      20.000000     0.000000 -0.910000     2.0          2.0       1.600000           4.200000
          3         5.0      40.000000    20.000000 -1.806000     4.0          2.0       2.600000           4.400000
          4         7.0      57.142857    14.285714 -2.284286     4.0          2.0       2.285714           3.428571
          5        10.0       0.000000     0.000000 -0.395000     5.0          2.0       4.300000           7.100000


## 6. Verify outputs (CHIRPS, NDVI, WFP)

Sanity-check shape and value ranges of the three files we built last session.

In [21]:
chirps = pd.read_csv("../../data/raw/chirps/chirps_sector_monthly.csv")
ndvi   = pd.read_csv("../../data/raw/ndvi/ndvi_sector_monthly.csv")
wfp    = pd.read_csv("../../data/raw/wfp/wfp_district_monthly.csv")

for name, d in [("CHIRPS", chirps), ("NDVI", ndvi), ("WFP", wfp)]:
    print(f"=== {name} === {d.shape[0]:,} rows × {d.shape[1]} cols")

print(f"\nCHIRPS rainfall_mm: min={chirps['rainfall_mm'].min():.1f}, "
      f"max={chirps['rainfall_mm'].max():.1f}, mean={chirps['rainfall_mm'].mean():.1f}")
print(f"NDVI:               min={ndvi['ndvi'].min():.3f}, "
      f"max={ndvi['ndvi'].max():.3f}, mean={ndvi['ndvi'].mean():.3f}")
print(f"WFP price_rwf:      min={wfp['price_rwf'].min():.0f}, "
      f"max={wfp['price_rwf'].max():.0f}, median={wfp['price_rwf'].median():.0f}")
print(f"\nWFP coverage: {wfp['admin2'].nunique()} of 30 districts, "
      f"{wfp['commodity'].nunique()} commodities")

=== CHIRPS === 30,384 rows × 7 cols
=== NDVI === 30,384 rows × 7 cols
=== WFP === 2,253 rows × 9 cols

CHIRPS rainfall_mm: min=0.0, max=417.2, mean=98.3
NDVI:               min=-0.013, max=0.904, mean=0.611
WFP price_rwf:      min=152, max=1492, median=538

WFP coverage: 7 of 30 districts, 6 commodities


## 7. Save cluster aggregate & summary

Persist DHS cluster-level data so notebook 02 doesn't re-run the whole DHS load.

In [22]:
OUTPUT_DIR = Path("../../data/processed/dhs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "dhs_cluster_2019_20.csv"
cluster_agg.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Saved: {OUTPUT_PATH.resolve()}")
print(f"   {len(cluster_agg)} clusters, {OUTPUT_PATH.stat().st_size / 1e3:.1f} KB")
print()
print("Notebook 01 complete. Next: 02_feature_engineering.ipynb")
print("  - Spatial join DHS clusters → GADM sectors (via RWGE81FL.zip GPS file)")
print("  - Build village-month master dataset (DHS + CHIRPS + NDVI + WFP + GADM)")
print("  - Engineer lag, anomaly, seasonality features")

✅ Saved: C:\Users\Hello\kurinda\data\processed\dhs\dhs_cluster_2019_20.csv
   500 clusters, 40.7 KB

Notebook 01 complete. Next: 02_feature_engineering.ipynb
  - Spatial join DHS clusters → GADM sectors (via RWGE81FL.zip GPS file)
  - Build village-month master dataset (DHS + CHIRPS + NDVI + WFP + GADM)
  - Engineer lag, anomaly, seasonality features
